In [1]:
import os, glob, ast
from tqdm import tqdm
import numpy as np
import pandas as pd
from PIL import Image

import torch, torch.nn as nn
from torch.utils.data import Dataset, DataLoader, TensorDataset
import torchvision.transforms as T
import timm

from sklearn.cluster import KMeans
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import normalized_mutual_info_score as NMI
from sklearn.metrics import adjusted_rand_score as ARI
from scipy.optimize import linear_sum_assignment

In [4]:
if not os.path.exists("labels_combined.csv"):
    all_dfs = []
    for csv_path in glob.glob("/kaggle/input/mlrs-net/labels/*.csv"):
        clsname = os.path.splitext(os.path.basename(csv_path))[0]  
        df = pd.read_csv(csv_path)
        if 'labels' in df.columns:
            df['labels'] = df['labels'].apply(ast.literal_eval)
        else:
            pass
        df['scene'] = clsname
        all_dfs.append(df)
    labels_df = pd.concat(all_dfs, ignore_index=True)
    labels_df.to_csv("labels_combined.csv", index=False)
else:
    labels_df = pd.read_csv("labels_combined.csv")
    if labels_df['labels'].dtype == object and isinstance(labels_df['labels'].iloc[0], str):
        labels_df['labels'] = labels_df['labels'].apply(ast.literal_eval)

print("Total images:", len(labels_df))
print("Unique scene classes:", labels_df['scene'].nunique())


Total images: 109161
Unique scene classes: 46


In [5]:
scene_classes = sorted(labels_df['scene'].unique().tolist())
num_known = int(0.6 * len(scene_classes))
known_scenes = scene_classes[:num_known]
labels_df['is_labeled'] = labels_df['scene'].isin(known_scenes)

labeled_df = labels_df[labels_df['is_labeled']].reset_index(drop=True)
unlabeled_df = labels_df[~labels_df['is_labeled']].reset_index(drop=True)

print("Known scene count:", len(known_scenes))
print("Labeled images:", len(labeled_df), "Unlabeled images:", len(unlabeled_df))

Known scene count: 27
Labeled images: 64599 Unlabeled images: 44562


In [6]:
class ImgDataset(Dataset):
    def __init__(self, df, img_root="images", transform=None):
        self.df = df.reset_index(drop=True)
        self.img_root = img_root
        self.transform = transform
        self.id2path = {}
        for scene in os.listdir(self.img_root):
            scene_dir = os.path.join(self.img_root, scene)
            if not os.path.isdir(scene_dir): continue
            for fname in os.listdir(scene_dir):
                name, ext = os.path.splitext(fname)
                self.id2path[name] = os.path.join(scene_dir, fname)

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        img_id = self.df.iloc[idx]['image_id']
        if img_id not in self.id2path:
            raise FileNotFoundError(f"{img_id} not found under {self.img_root}")
        path = self.id2path[img_id]
        img = Image.open(path).convert("RGB")
        if self.transform: img = self.transform(img)
        return img, img_id

transform = T.Compose([
    T.Resize((224, 224)),
    T.ToTensor(),
    T.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])


In [8]:
ds_all = ImgDataset(labels_df, img_root="/kaggle/input/mlrs-net/images", transform=transform)
loader_all = DataLoader(ds_all, batch_size=64, shuffle=False, num_workers=2, pin_memory=True)

In [9]:
device = "cuda" if torch.cuda.is_available() else "cpu"
try:
    backbone = timm.create_model('vit_base_patch16_224_dino', pretrained=True, num_classes=0)  # num_classes=0 -> features
except Exception as e:
    # fallback to vit_base_patch16_224 and hope pretrained weights are DINO or ImageNet
    print("Warning: exact DINO model name not found in timm; trying 'vit_base_patch16_224' pretrained")
    backbone = timm.create_model('vit_base_patch16_224', pretrained=True, num_classes=0)

backbone = backbone.to(device).eval()
print("Backbone loaded. Model type:", type(backbone))

/usr/local/lib/python3.11/dist-packages/timm/models/_factory.py:126: UserWarning: Mapping deprecated model name vit_base_patch16_224_dino to current vit_base_patch16_224.dino.
  model = create_fn(


model.safetensors:   0%|          | 0.00/343M [00:00<?, ?B/s]

Backbone loaded. Model type: <class 'timm.models.vision_transformer.VisionTransformer'>


In [10]:
all_feats = []
all_ids = []
with torch.no_grad():
    for imgs, ids in tqdm(loader_all, desc="Extracting features"):
        imgs = imgs.to(device)
        feats = backbone(imgs)        
        feats = feats.cpu().numpy()
        all_feats.append(feats)
        all_ids.extend(ids)
all_feats = np.vstack(all_feats)
print("Features extracted:", all_feats.shape)

Extracting features: 100%|██████████| 1706/1706 [21:25<00:00,  1.33it/s]

Features extracted: (109161, 768)


In [11]:
id2idx = {img_id: i for i, img_id in enumerate(all_ids)}

In [12]:
labeled_ids = labeled_df['image_id'].tolist()
l_indices = [id2idx[i] for i in labeled_ids]
X_labeled = all_feats[l_indices]
y_labeled = labeled_df['scene'].values

In [13]:
unlabeled_ids = unlabeled_df['image_id'].tolist()
u_indices = [id2idx[i] for i in unlabeled_ids]
X_unl = all_feats[u_indices]

In [14]:
novel_scene_count = unlabeled_df['scene'].nunique()
K = novel_scene_count
print("Novel scene count (K):", K)

Novel scene count (K): 19


In [15]:
from collections import defaultdict
centroids = []
centroid_labels = []
for cls in np.unique(y_labeled):
    idxs = [i for i,lab in enumerate(y_labeled) if lab==lab] 
known_classes = np.unique(y_labeled)
for cls in known_classes:
    idxs = [i for i,lab in enumerate(y_labeled) if lab==cls]
    c = X_labeled[idxs].mean(axis=0)
    centroids.append(c)
    centroid_labels.append(cls)

In [16]:
kmeans_unl = KMeans(n_clusters=K, random_state=42, n_init=20).fit(X_unl)
for k in range(K):
    centroids.append(kmeans_unl.cluster_centers_[k])
    centroid_labels.append(f"novel_{k}")

centroids = np.vstack(centroids)  # shape (C, D)
print("Total centroids (known + novel):", centroids.shape[0])


Total centroids (known + novel): 46


In [19]:
from sklearn.metrics import pairwise_distances_argmin
nearest = pairwise_distances_argmin(all_feats, centroids)
assigned = [centroid_labels[i] for i in nearest]

distances = np.linalg.norm(all_feats - centroids[nearest], axis=1)


In [20]:
pseudo_pairs = []
keep_thresh = 0.7  
for k in range(K):
    idxs = [i for i,v in enumerate(nearest) if v == (len(known_classes) + k)]
    if len(idxs) == 0: continue
    dists = distances[idxs]
    thr = np.quantile(dists, keep_thresh)  
    for j, idx in enumerate(idxs):
        if distances[idx] <= thr:
            img_id = all_ids[idx]
            pseudo_pairs.append((img_id, f"novel_{k}"))

pseudo_df = pd.DataFrame(pseudo_pairs, columns=['image_id', 'pseudo_label'])
print("Pseudo-labeled examples:", len(pseudo_df))

Pseudo-labeled examples: 33495


In [21]:
train_known = labeled_df[['image_id','scene']].rename(columns={'scene':'label'})
train_pseudo = pseudo_df.rename(columns={'pseudo_label':'label'})
train_combined = pd.concat([train_known, train_pseudo], ignore_index=True)
train_combined = train_combined.sample(frac=1, random_state=42).reset_index(drop=True)

le = LabelEncoder()
train_combined['label_idx'] = le.fit_transform(train_combined['label'])
num_final_classes = len(le.classes_)
print("Training classes (known + pseudo-novel):", num_final_classes)
print("Example classes:", le.classes_[:10])

Training classes (known + pseudo-novel): 46
Example classes: ['airplane' 'airport' 'bareland' 'baseball_diamond' 'basketball_court'
 'beach' 'bridge' 'chaparral' 'cloud' 'commercial_area']


In [23]:
train_X = np.stack([ all_feats[id2idx[i]] for i in train_combined['image_id'] ])
train_y = train_combined['label_idx'].values

train_dataset = TensorDataset(torch.from_numpy(train_X).float(), torch.from_numpy(train_y).long())
train_loader = DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)

clf = nn.Linear(train_X.shape[1], num_final_classes).to(device)
opt = torch.optim.AdamW(clf.parameters(), lr=1e-3)
lossfn = nn.CrossEntropyLoss()

EPOCHS = 10
for epoch in range(EPOCHS):
    clf.train()
    running = 0.0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        out = clf(xb)
        loss = lossfn(out, yb)
        loss.backward(); opt.step()
        running += loss.item() * xb.size(0)
    print(f"Epoch {epoch+1}/{EPOCHS} train loss: {running/len(train_dataset):.4f}")

Epoch 1/10 train loss: 0.4020
Epoch 2/10 train loss: 0.2777
Epoch 3/10 train loss: 0.2541
Epoch 4/10 train loss: 0.2441
Epoch 5/10 train loss: 0.2388
Epoch 6/10 train loss: 0.2346
Epoch 7/10 train loss: 0.2294
Epoch 8/10 train loss: 0.2237
Epoch 9/10 train loss: 0.2238
Epoch 10/10 train loss: 0.2189


In [24]:
true_labels = labels_df['scene'].values

pred_labels = np.array(assigned)

pred_le = LabelEncoder()
pred_int = pred_le.fit_transform(pred_labels)
true_le = LabelEncoder()
true_int = true_le.fit_transform(true_labels)


In [25]:
def clustering_accuracy(y_true, y_pred):
    # y_true, y_pred are integer arrays
    D = max(y_pred.max(), y_true.max()) + 1
    w = np.zeros((D, D), dtype=np.int64)
    for i in range(y_pred.size):
        w[y_pred[i], y_true[i]] += 1
    row_ind, col_ind = linear_sum_assignment(w.max() - w)
    return w[row_ind, col_ind].sum() / y_pred.size

acc = clustering_accuracy(true_int, pred_int)
nmi = NMI(true_int, pred_int)
ari = ARI(true_int, pred_int)
print(f"Clustering ACC (Hungarian): {acc:.4f} | NMI: {nmi:.4f} | ARI: {ari:.4f}")

Clustering ACC (Hungarian): 0.7361 | NMI: 0.7300 | ARI: 0.5706


In [26]:
np.save("all_feats.npy", all_feats)
pd.DataFrame({'image_id': all_ids, 'assigned_cluster': assigned, 'distance': distances}).to_csv("assigned_clusters.csv", index=False)
train_combined.to_csv("train_combined_labels.csv", index=False)
print("Saved features and assignments.")

Saved features and assignments.
